In [87]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from art import tprint
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [88]:
data = pd.read_csv('UDG-CA-1067-3.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4460 entries, 0 to 4459
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Date        4460 non-null   object 
 1   Time        4460 non-null   object 
 2   Temp        4460 non-null   object 
 3   Hum         4460 non-null   object 
 4   Wind Speed  4460 non-null   float64
 5   Presureh    4460 non-null   object 
 6   Solar Rad.  4460 non-null   object 
 7   ET          4460 non-null   float64
dtypes: float64(2), object(6)
memory usage: 278.9+ KB


In [89]:
# 1. Limpiar valores no numéricos
data = data.replace('---', np.nan)  # Reemplazar '---' con NaN
data = data.dropna()
data = data.astype(float, errors='ignore')  # Asegurar que todas las columnas sean numéricas

In [90]:
data["DateTime"] = pd.to_datetime(data["Date"] + " " + data["Time"])
data = data.drop(["Date", "Time", "DateTime"], axis=1)

C:\Users\compa\AppData\Local\Temp\ipykernel_2480\865406847.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["DateTime"] = pd.to_datetime(data["Date"] + " " + data["Time"])


In [91]:
# Separar las features (X) del la variable Y (y)
X = data.drop("ET", axis=1).values
y = data["ET"].values

In [92]:
# Normalizar las características
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

In [93]:
# Dividir en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [94]:
# 2. Crear el MLP
model = Sequential([
    Dense(64, input_dim=X.shape[1], activation='relu'),  # Capa oculta 1
    Dense(32, activation='relu'),                        # Capa oculta 2
    Dense(1)                                             # Capa de salida
])

In [95]:
# Compilar el modelo
model.compile(optimizer='adam', loss='mse', metrics=['mae'])    

In [96]:
# 3. Entrenar el modelo
model.fit(X_train, y_train, epochs=100, batch_size=1600, validation_data=(X_test, y_test))

Epoch 1/100
3/3 [==============================] - 1s 112ms/step - loss: 0.0293 - mae: 0.1563 - val_loss: 0.0148 - val_mae: 0.0833
Epoch 2/100
3/3 [==============================] - 0s 24ms/step - loss: 0.0140 - mae: 0.0680 - val_loss: 0.0136 - val_mae: 0.0484
Epoch 3/100
3/3 [==============================] - 0s 25ms/step - loss: 0.0146 - mae: 0.0550 - val_loss: 0.0151 - val_mae: 0.0651
Epoch 4/100
3/3 [==============================] - 0s 25ms/step - loss: 0.0152 - mae: 0.0659 - val_loss: 0.0130 - val_mae: 0.0605
Epoch 5/100
3/3 [==============================] - 0s 24ms/step - loss: 0.0129 - mae: 0.0607 - val_loss: 0.0114 - val_mae: 0.0578
Epoch 6/100
3/3 [==============================] - 0s 24ms/step - loss: 0.0118 - mae: 0.0595 - val_loss: 0.0118 - val_mae: 0.0622
Epoch 7/100
3/3 [==============================] - 0s 22ms/step - loss: 0.0123 - mae: 0.0649 - val_loss: 0.0122 - val_mae: 0.0671
Epoch 8/100
3/3 [==============================] - 0s 26ms/step - loss: 0.0125 - mae: 0.0

In [97]:
# Evaluar el modelo
loss, mae = model.evaluate(X_test, y_test)
tprint(f"Perdida: {loss}, MAE: {mae}")

28/28 [==============================] - 0s 2ms/step - loss: 0.0109 - mae: 0.0450
 ____                   _  _      _              ___       ___   _   ___    ___    ___   _  _    _  ____    ___    __     __    _____  ____   _  _    _   ___    __    _____      __  __     _     _____       ___       ___   _  _    ____    ___    ___   ____   _  ____   _  _  _    _____   ___   _____  _   ___   _  _____ 
|  _ \   ___  _ __   __| |(_)  __| |  __ _  _   / _ \     / _ \ / | / _ \  / _ \  / _ \ | || |  / ||___ \  ( _ )  / /_   / /_  |___ / |___ \ | || |  / | ( _ )  / /_  |___ /     |  \/  |   / \   | ____| _   / _ \     / _ \ | || |  | ___|  / _ \  / _ \ |___ \ / ||___ \ / || || |  |___  | ( _ ) |___ / / | / _ \ / ||___  |
| |_) | / _ \| '__| / _` || | / _` | / _` |(_) | | | |   | | | || || | | || (_) || | | || || |_ | |  __) | / _ \ | '_ \ | '_ \   |_ \   __) || || |_ | | / _ \ | '_ \   |_ \     | |\/| |  / _ \  |  _|  (_) | | | |   | | | || || |_ |___ \ | | | || | | |  __) || |  __) || || || 

In [98]:
# Predecir ET con datos nuevos
predictions = model.predict(X_test)
print("Predicciones:", predictions)

28/28 [==============================] - 0s 2ms/step
Predicciones: [[ 7.76558183e-04]
 [ 2.04700138e-03]
 [ 1.97100546e-03]
 [ 7.91945960e-03]
 [ 3.66135687e-02]
 [ 3.86644807e-03]
 [ 1.09968986e-03]
 [ 2.50738021e-03]
 [ 6.21003564e-03]
 [ 3.33674904e-03]
 [ 2.84473505e-03]
 [ 4.55750786e-02]
 [ 5.91831189e-03]
 [ 1.80957913e-02]
 [ 3.60481851e-02]
 [ 1.63029786e-03]
 [ 8.41730740e-03]
 [ 9.43702366e-03]
 [ 8.71922821e-02]
 [-8.86850990e-04]
 [ 1.11097749e-03]
 [ 1.81538519e-03]
 [-7.04401825e-03]
 [ 9.18905251e-04]
 [ 5.16390726e-02]
 [ 1.16572520e-02]
 [ 8.62145424e-02]
 [ 3.85879539e-04]
 [ 1.87170804e-02]
 [ 6.11131564e-02]
 [ 1.06923794e-02]
 [ 9.40357000e-02]
 [ 4.49935999e-03]
 [ 8.33401829e-02]
 [ 9.41021275e-03]
 [ 1.40335247e-01]
 [ 3.12176254e-03]
 [ 8.50009732e-04]
 [ 8.42423439e-02]
 [ 4.98730727e-02]
 [ 1.34035852e-03]
 [ 3.02998386e-02]
 [ 1.28049310e-03]
 [ 3.02355643e-03]
 [ 2.11591274e-02]
 [ 6.27848413e-03]
 [-5.44889364e-03]
 [ 8.01689252e-02]
 [ 2.18484327e-02]
 [

In [100]:
# Calcular MSE (Mean Squared Error)
mse = mean_squared_error(y_test, predictions)
print(f'MSE (Mean Squared Error): {mse}')

# Calcular MAE (Mean Absolute Error)
mae = mean_absolute_error(y_test, predictions)
print(f'MAE (Mean Absolute Error): {mae}')

# Calcular R^2 (Coeficiente de determinación)
r2 = r2_score(y_test, predictions)
print(f'R² (Coeficiente de determinación): {r2}')


MSE (Mean Squared Error): 0.010904126777762117
MAE (Mean Absolute Error): 0.0450021213510245
R² (Coeficiente de determinación): 0.08525487819898736


# README
1. Preprocesamiento
-   Los valores no numéricos ('---') se reemplazan con NaN.
-   Se eliminan o cambian valores faltantes para asegurar que los datos sean utilizables.
-   Las características (Temp, Hum, etc.) se normalizan utilizando MinMaxScaler para que estén en un rango de [0, 1].

2. Modelo MLP
Se define un MLP con:
-       64 neuronas en la primera capa oculta.
-       32 neuronas en la segunda capa oculta.
-       1 neurona en la capa de salida (para la predicción de ET).
Se utiliza el optimizador Adam y la función de pérdida MSE (Mean Squared Error).

3. Entrenamiento y Evaluación
-   El modelo se entrena durante 50 épocas con un tamaño de lote de 16.
-   Durante el entrenamiento, se mide la métrica MAE (Mean Absolute Error) para evaluar el desempeño.

4. Resultados
-   Pérdida (Loss): Error cuadrático medio en los datos de prueba.
-   MAE: Error absoluto medio, una métrica interpretativa para problemas de regresión.
-   Predicciones: El modelo predice valores de ET basándose en las características de entrada.
